# Mastery-triggered curriculum — analysis

Compare fixed-late, fixed-early, and performance-triggered stage transitions under the same 9,000-update, constant-LR budget. The gate uses a scene-disjoint **progress-dev** subset; the main analysis uses **heldout-dev**. No model training occurs in this notebook.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

path = PROJECT_ROOT / 'outputs/medium_world/experiments/mastery_progression/schedule_comparison.json'
report = json.loads(path.read_text(encoding='utf-8'))
runs = report['runs']
POLICIES = ('fixed_late', 'fixed_early', 'mastery')
TASKS = ('color', 'shape', 'conjunction', 'pair')
print('Seeds:', sorted({r['seed'] for r in runs}), '| Updates:', report['config']['total_updates'])
print('Attribute targets:', report['config']['attribute_targets'])
print('Conjunction targets:', report['config']['conjunction_targets'])

## 1. Did advancement happen through mastery or a stage cap?

Reaching the maximum permitted stage duration is **not** equivalent to satisfying the performance criterion. We therefore begin with the actual transition step and reason for each seed, before looking at accuracy.

In [ ]:
transitions = pd.DataFrame([{
    'seed': r['seed'], 'policy': r['policy'],
    'attributes_end': r['transitions'][0]['step'],
    'attributes_reason': r['transitions'][0]['reason'],
    'conjunction_end': r['transitions'][1]['step'],
    'conjunction_reason': r['transitions'][1]['reason'],
    'common_mixed': r['transitions'][2]['step']
} for r in runs])
display(transitions.sort_values(['seed', 'policy']).reset_index(drop=True))
display(transitions[transitions.policy == 'mastery']
        [['attributes_reason', 'conjunction_reason']].apply(pd.Series.value_counts).fillna(0))

## 2. What was the model able to demonstrate at transition?

The gate averages the past three progress-dev evaluations and requires two qualifying checks. The targets are researcher-defined, and success on a small progress set is not a guarantee of out-of-sample competence. We display single-checkpoint progress and held-out performance at each transition; **these are not the smoothed threshold values**.

In [ ]:
gate_rows = []
for r in runs:
    if r['policy'] != 'mastery':
        continue
    for tr in r['transitions'][:2]:
        h = next(h for h in r['history'] if h['step'] == tr['step'])
        gate_rows.append({
            'seed': r['seed'], 'stage': tr['from'], 'step': tr['step'],
            'reason': tr['reason'],
            'shape_progress': 100 * h['progress_dev']['by_template']['shape'],
            'shape_heldout': 100 * h['heldout_dev']['by_template']['shape'],
            'color_progress': 100 * h['progress_dev']['by_template']['color'],
            'conjunction_progress': 100 * h['progress_dev']['by_template']['conjunction']
        })
display(pd.DataFrame(gate_rows).sort_values(['stage', 'seed']).round(2))

## 3. Final generalization and learning-curve area

The primary final metric is **held-out ID binding accuracy**, not the progress set used to choose transitions. AULC measures average accuracy during the whole 9,000-step budget:

\[\mathrm{AULC}=\frac{1}{T}\sum_i\frac{A(t_i)+A(t_{i-1})}{2}(t_i-t_{i-1}).\]

The development split is scene-disjoint but not task-label-stratified, so also inspect the full ID-dev and balanced OOD-dev results. OOD BCE measures the probability assigned to correct answers, not just the decision accuracy.

In [ ]:
def aulc(history, split='heldout_dev', task='pair'):
    steps = np.array([h['step'] for h in history], dtype=float)
    acc = np.array([h[split]['by_template'][task] for h in history])
    return np.sum(np.diff(steps) * (acc[1:] + acc[:-1]) / 2) / (steps[-1] - steps[0])

rows = []
for r in runs:
    f = r['final']
    rows.append({
        'seed': r['seed'], 'policy': r['policy'],
        'heldout_pair': 100 * f['heldout_dev']['by_template']['pair'],
        'id_pair': 100 * f['id_dev']['by_template']['pair'],
        'ood_pair': 100 * f['ood_dev']['by_template']['pair'],
        'ood_bce': f['ood_dev']['loss'],
        'heldout_overall': 100 * f['heldout_dev']['accuracy'],
        'shape': 100 * f['heldout_dev']['by_template']['shape'],
        'color': 100 * f['heldout_dev']['by_template']['color'],
        'conjunction': 100 * f['heldout_dev']['by_template']['conjunction'],
        'heldout_pair_aulc': 100 * aulc(r['history']),
        'id_pair_aulc': 100 * r['id_pair_aulc']
    })
final = pd.DataFrame(rows)
metrics = ['heldout_pair', 'id_pair', 'ood_pair', 'ood_bce',
           'heldout_pair_aulc', 'heldout_overall', 'shape']
display(final.sort_values(['seed', 'policy']).reset_index(drop=True).round(2))
display(final.groupby('policy')[metrics].agg(['mean', 'std']).reindex(POLICIES).round(2))

## 4. Do paired seeds favor a mastery-based decision?

For paired seed \(s\), \(\Delta_s=A_s(\mathrm{mastery})-A_s(\mathrm{fixed})\). Positive differences favor mastery; report the number of favorable seeds as well as the mean. A small advantage across five seeds does not justify a universal claim.

In [ ]:
comparisons = (('mastery', 'fixed_early'), ('mastery', 'fixed_late'),
               ('fixed_early', 'fixed_late'))
differences = []
for first, second in comparisons:
    for metric in ('heldout_pair', 'id_pair', 'ood_pair', 'heldout_pair_aulc',
                   'heldout_overall', 'shape', 'ood_bce'):
        pivot = final.pivot(index='seed', columns='policy', values=metric)
        delta = pivot[first] - pivot[second]
        differences.append({'contrast': first + ' − ' + second, 'metric': metric,
                            'mean_difference': delta.mean(), 'sd': delta.std(),
                            'wins': (delta > 0).sum()})
display(pd.DataFrame(differences).round(3))

## 5. When did binding and shape accuracy improve?

A fixed-early schedule gives the model more updates **after** introducing binding. These curves plot performance against total optimizer steps, not examples spent specifically on binding. We display held-out binding and shape, and print the mean transition times to interpret the timing shift.

In [ ]:
curves = pd.DataFrame([{
    'seed': r['seed'], 'policy': r['policy'], 'step': h['step'],
    'pair': 100 * h['heldout_dev']['by_template']['pair'],
    'shape': 100 * h['heldout_dev']['by_template']['shape']
} for r in runs for h in r['history']])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ('pair', 'shape')):
    for policy in POLICIES:
        curve = curves[curves.policy == policy].groupby('step')[metric].agg(['mean', 'std'])
        ax.plot(curve.index, curve['mean'], label=policy)
        ax.fill_between(curve.index, curve['mean'] - curve['std'],
                        curve['mean'] + curve['std'], alpha=.10)
    ax.set(title='Held-out ID ' + metric, xlabel='Optimizer updates',
           ylabel='Accuracy (%)')
    ax.legend()
plt.tight_layout()
plt.show()
display(transitions.groupby('policy')[['attributes_end', 'conjunction_end']].mean())

## 6. Does quota-based rehearsal remain available?

Equal **total** presentations do not guarantee rehearsal during every stage. If the earlier-task quotas are depleted before the binding stage, binding may become a single-task phase. We measure actual presentation counts rather than assuming replay always occurred.

In [ ]:
exposures = []
for r in runs:
    for phase, counts in r['phase_task_presentations'].items():
        total = sum(counts.values())
        replay = total - counts.get('pair', 0) if phase == 'binding' else np.nan
        exposures.append({
            'seed': r['seed'], 'policy': r['policy'], 'phase': phase,
            'total_examples': total, 'pair_examples': counts.get('pair', 0),
            'rehearsal_examples': replay,
            'rehearsal_share': 100 * replay / total if phase == 'binding' else np.nan
        })
exposure = pd.DataFrame(exposures)
display(exposure[exposure.phase == 'binding']
        .sort_values(['seed', 'policy']).reset_index(drop=True).round(2))

## 7. Temporary forgetting and common consolidation

Compare skills when binding is first introduced, just before the common mixed stage (step 7,200), and at the end (step 9,000). These changes describe forgetting/recovery during different task mixtures, **not** a controlled rehearsal-versus-no-rehearsal ablation.

In [ ]:
retention = []
for r in runs:
    checkpoints = {h['step']: h for h in r['history']}
    binding_start = r['transitions'][1]['step']
    for task in TASKS:
        begin = 100 * checkpoints[binding_start]['heldout_dev']['by_template'][task]
        before_common = 100 * checkpoints[7200]['heldout_dev']['by_template'][task]
        end = 100 * checkpoints[9000]['heldout_dev']['by_template'][task]
        retention.append({
            'seed': r['seed'], 'policy': r['policy'], 'task': task,
            'during_binding': before_common - begin,
            'during_consolidation': end - before_common
        })
retention = pd.DataFrame(retention)
display(retention.groupby(['policy', 'task'])
        [['during_binding', 'during_consolidation']].mean().round(2))

## What this study actually supports

We can establish whether the gate **triggered**, whether earlier binding exposure improved learning-curve efficiency, and whether older tasks temporarily declined. We cannot infer general compositional understanding from aggregate accuracy alone. The progress set was consulted repeatedly, and OOD-dev remains an exploratory split rather than a fresh final test. The next step is prediction-level shape and binding error analysis, not another arbitrary threshold sweep.